Kafka (EC2) → Databricks (PySpark Streaming) → 
    → S3 (raw data)
    → DynamoDB (aggregated data)

1. Read Data from Kafka (Streaming)

In [ ]:
df = (
    spark.readStream
        .format("kafka")
        .option("kafka.bootstrap.servers", "43.205.136.148:9092")
        .option("subscribe", "test-topic")
        .option("startingOffsets", "earliest")
        .load()
)

🔍 Explanation:

readStream → Enables real-time streaming

format("kafka") → Connects Spark to Kafka

bootstrap.servers → Your EC2 Kafka IP

subscribe → Topic name

startingOffsets=earliest → Reads all past + new data

👉 Output schema (important):

key | value | topic | partition | offset | timestamp

2. Convert Kafka Value to JSON String

In [ ]:
from pyspark.sql.functions import col

parsed_df = df.selectExpr("CAST(value AS STRING) as json_data")

🔍 Explanation:

Kafka sends data in binary format → Spark converts it to string.

3. Define Schema and Parse JSON

In [ ]:
from pyspark.sql.functions import from_json
from pyspark.sql.types import *

schema = StructType([
    StructField("sale_id", IntegerType()),
    StructField("order_id", StringType()),
    StructField("customer_id", StringType()),
    StructField("product_id", StringType()),
    StructField("product_name", StringType()),
    StructField("category", StringType()),
    StructField("quantity", IntegerType()),
    StructField("unit_price", DoubleType()),
    StructField("total_amount", DoubleType()),
    StructField("sale_date", StringType()),
    StructField("payment_method", StringType()),
    StructField("region", StringType()),
    StructField("created_at", StringType())
])

json_df = parsed_df.select(
    from_json(col("json_data"), schema).alias("data")
).select("data.*")

🔍 Explanation:

from_json() → Converts JSON string into structured columns

Schema ensures proper data types

data.* → flattens JSON

🔷 4. Write Raw Data to S3 (Bronze Layer)

(You used workaround due to Community Edition)

In [ ]:
json_df.write.mode("overwrite").json("/tmp/kafka_raw_data")

🔍 Explanation:

Writes raw Kafka data

This is your Bronze layer

Stored locally (since S3 direct write not allowed)

🔷 5. Aggregate Data (Transformation - Silver Layer)

In [ ]:
agg_df = json_df.groupBy("product_id", "product_name", "category") \
    .sum("quantity", "total_amount") \
    .withColumnRenamed("sum(quantity)", "total_quantity") \
    .withColumnRenamed("sum(total_amount)", "total_revenue")

🔍 Explanation:

Groups data by product

Calculates:

total quantity sold

total revenue

👉 This becomes your analytics dataset

🔷 6. Split Data by Category

In [ ]:
electronics_df = agg_df.filter("category = 'Electronics'")
accessories_df = agg_df.filter("category = 'Accessories'")
furniture_df = agg_df.filter("category = 'Furniture'")

🔍 Explanation:

Databricks CE cannot write directly to DynamoDB

So we convert Spark → Pandas → boto3

🔷 8. Setup DynamoDB Connection

In [ ]:
import boto3

dynamodb = boto3.resource(
    'dynamodb',
    aws_access_key_id="YOUR_ACCESS_KEY",
    aws_secret_access_key="YOUR_SECRET_KEY",
    region_name="ap-south-1"
)

🔷 9. Handle Data Types (Critical Fix)

In [ ]:
from decimal import Decimal
import datetime

def convert_value(value):
    if isinstance(value, float):
        return Decimal(str(value))
    elif isinstance(value, (datetime.date, datetime.datetime)):
        return str(value)
    else:
        return value

def insert_into_dynamodb(table_name, dataframe):
    table = dynamodb.Table(table_name)

    for _, row in dataframe.iterrows():
        item = {k: convert_value(v) for k, v in row.to_dict().items()}
        table.put_item(Item=item)

In [ ]:
def insert_into_dynamodb(table_name, dataframe):
    table = dynamodb.Table(table_name)

    for _, row in dataframe.iterrows():
        item = {k: convert_value(v) for k, v in row.to_dict().items()}
        table.put_item(Item=item)

11. Push Data to Tables

In [ ]:
insert_into_dynamodb("Electronics", electronics_pd)
insert_into_dynamodb("Accessories", accessories_pd)
insert_into_dynamodb("Furniture", furniture_pd)

🔷 12. Final Output

You achieved:

Kafka → Streaming → Structured Data → Aggregation → DynamoDB